In [ ]:
# ============================================================
# 02_final_cohort_definition_and_baseline_characterization
# Thesis final pipeline
# ============================================================

import sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)

PROJECT_CONSTANTS_DIR = Path(
    "/home/azureuser/cloudfiles/code/Users/m.bolognini/"
    "UseCase-Code/data-science/projects/a_MB_thesis_final/pipeline_thesis_final"
)

sys.path.append(str(PROJECT_CONSTANTS_DIR))

from project_constants import *

print("OUTPUT_01:", OUTPUT_01)
print("OUTPUT_02:", OUTPUT_02)

In [ ]:
# ============================================================
# STUDY CONSTANTS QC
# ============================================================

print("STUDY_START:", STUDY_START)
print("STUDY_END:", STUDY_END)
print("FOLLOWUP_DATE:", FOLLOWUP_DATE)

print("COHORT1_START:", COHORT1_START)
print("COHORT1_END:", COHORT1_END)
print("COHORT2_START:", COHORT2_START)
print("COHORT2_END:", COHORT2_END)

print("PALLIATIVE_DISCHARGE_CODES:", PALLIATIVE_DISCHARGE_CODES)

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def n_pct(series):
    s = series.dropna()
    if len(s) == 0:
        return "NA"
    n = int((s == 1).sum())
    total = len(s)
    pct = 100 * n / total
    return f"{n}/{total} ({pct:.1f}%)"


def median_iqr(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return "NA"
    return f"{s.median():.1f} ({s.quantile(0.25):.1f}–{s.quantile(0.75):.1f})"


def categorical_n_pct(series, category):
    s = series.dropna()
    if len(s) == 0:
        return "NA"
    n = int((s == category).sum())
    total = len(s)
    pct = 100 * n / total
    return f"{n} ({pct:.1f}%)"


def compare_continuous(df, variable, group_col):
    tmp = df[[variable, group_col]].dropna().copy()
    tmp[variable] = pd.to_numeric(tmp[variable], errors="coerce")
    tmp = tmp.dropna()

    groups = list(tmp[group_col].dropna().unique())
    if len(groups) != 2:
        return np.nan

    g0 = tmp.loc[tmp[group_col] == groups[0], variable]
    g1 = tmp.loc[tmp[group_col] == groups[1], variable]

    if len(g0) < 3 or len(g1) < 3:
        return np.nan

    try:
        _, p = mannwhitneyu(g0, g1, alternative="two-sided")
        return p
    except Exception:
        return np.nan


def compare_categorical(df, variable, group_col):
    tmp = remove_duplicate_columns(df)[[variable, group_col]].dropna().copy()
    tmp[variable] = tmp[variable].astype("string")
    tmp[group_col] = tmp[group_col].astype("string")
    if tmp.empty:
        return np.nan

    table = pd.crosstab(tmp[variable], tmp[group_col])

    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan

    try:
        if table.shape == (2, 2) and (table.values < 5).any():
            _, p = fisher_exact(table)
        else:
            _, p, _, _ = chi2_contingency(table)
        return p
    except Exception:
        return np.nan

def remove_duplicate_columns(df):
    duplicated = df.columns[df.columns.duplicated()].tolist()
    if duplicated:
        print("Duplicated columns removed:", duplicated)
    return df.loc[:, ~df.columns.duplicated()].copy()

def format_p(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def smd_continuous(df, variable, group_col):
    tmp = df[[variable, group_col]].dropna().copy()
    tmp[variable] = pd.to_numeric(tmp[variable], errors="coerce")
    tmp = tmp.dropna()

    groups = list(tmp[group_col].dropna().unique())
    if len(groups) != 2:
        return np.nan

    x0 = tmp.loc[tmp[group_col] == groups[0], variable]
    x1 = tmp.loc[tmp[group_col] == groups[1], variable]

    if len(x0) < 2 or len(x1) < 2:
        return np.nan

    pooled_sd = np.sqrt((x0.var(ddof=1) + x1.var(ddof=1)) / 2)
    if pooled_sd == 0 or pd.isna(pooled_sd):
        return 0.0

    return (x1.mean() - x0.mean()) / pooled_sd


def smd_binary(df, variable, group_col, positive_value=1):
    tmp = df[[variable, group_col]].dropna().copy()
    groups = list(tmp[group_col].dropna().unique())
    if len(groups) != 2:
        return np.nan

    p0 = (tmp.loc[tmp[group_col] == groups[0], variable] == positive_value).mean()
    p1 = (tmp.loc[tmp[group_col] == groups[1], variable] == positive_value).mean()

    pooled = np.sqrt((p0 * (1 - p0) + p1 * (1 - p1)) / 2)
    if pooled == 0 or pd.isna(pooled):
        return 0.0

    return (p1 - p0) / pooled

In [ ]:
# ============================================================
# LOAD PREPARED DATASETS
# ============================================================

df_episode_master = pd.read_parquet(OUTPUT_01 / "01_episode_master.parquet")
df_clinical = pd.read_parquet(OUTPUT_01 / "01_clinical.parquet")
df_solid = pd.read_parquet(OUTPUT_01 / "01_solidTumor.parquet")
df_death = pd.read_parquet(OUTPUT_01 / "01_death.parquet")
df_lab_intensity = pd.read_parquet(OUTPUT_01 / "01_lab_measurement_intensity.parquet")

print("01_episode_master:", df_episode_master.shape)
print("01_clinical:", df_clinical.shape)
print("01_solidTumor:", df_solid.shape)
print("01_death:", df_death.shape)
print("01_lab_measurement_intensity:", df_lab_intensity.shape)

for df_ in [df_episode_master, df_clinical, df_solid, df_death, df_lab_intensity]:
    if "episode_key" in df_.columns:
        df_["episode_key"] = (
            df_["episode_key"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

In [ ]:
# ============================================================
# DATE CONVERSION AND BASIC CHECKS
# ============================================================

date_cols = [
    "startDate",
    "endDate",
    "ER_last_day",
    "ward_startDate",
    "dateOfDeath",
]

for col in date_cols:
    if col in df_episode_master.columns:
        df_episode_master[col] = pd.to_datetime(df_episode_master[col], errors="coerce")

print("Rows:", df_episode_master.shape[0])
print("Patients:", df_episode_master["pubID"].nunique())
print("Episodes:", df_episode_master["episode_key"].nunique())
print("Duplicated episode_key:", df_episode_master["episode_key"].duplicated().sum())
print("Deaths with date:", df_episode_master["dateOfDeath"].notna().sum())
print("Negative LOS:", (df_episode_master["endDate"] < df_episode_master["startDate"]).sum())

assert df_episode_master["episode_key"].duplicated().sum() == 0
assert (df_episode_master["endDate"] < df_episode_master["startDate"]).sum() == 0

In [ ]:
# ============================================================
# STUDY WINDOW AND WARD FILTER
# ============================================================

flow_rows = []

df0 = df_episode_master.copy()

flow_rows.append({
    "step": "Initial episode master",
    "episodes": df0.shape[0],
    "patients": df0["pubID"].nunique(),
})

df = df0[
    (df0["endDate"] >= STUDY_START) &
    (df0["endDate"] <= STUDY_END)
].copy()

flow_rows.append({
    "step": "After study window filter",
    "episodes": df.shape[0],
    "patients": df["pubID"].nunique(),
})

df["hospitalWard"] = (
    df["hospitalWard"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df = df[df["hospitalWard"].isin(VALID_WARDS)].copy()

flow_rows.append({
    "step": "After Internal Medicine A/B filter",
    "episodes": df.shape[0],
    "patients": df["pubID"].nunique(),
})

print("After study window and ward filtering:")
print("Rows:", df.shape[0])
print("Patients:", df["pubID"].nunique())
print("Episodes:", df["episode_key"].nunique())

In [ ]:
# ============================================================
# INDEX EPISODE DEFINITION
# ============================================================

# The unit of analysis is the last Internal Medicine hospitalization per patient
# within the study window. Earlier hospitalizations from the same patient are
# excluded to obtain one independent observation per patient and to simplify
# temporal validation and outcome attribution.

In [ ]:
# ============================================================
# LAST HOSPITALIZATION PER PATIENT
# ============================================================

df["pubID"] = (
    df["pubID"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df = df.sort_values(["pubID", "endDate", "episode_key"])

df_last = (
    df
    .groupby("pubID", as_index=False)
    .tail(1)
    .copy()
)

flow_rows.append({
    "step": "After last hospitalization per patient",
    "episodes": df_last.shape[0],
    "patients": df_last["pubID"].nunique(),
})

print("Final patient-level cohort:")
print("Rows:", df_last.shape[0])
print("Patients:", df_last["pubID"].nunique())
print("Episodes:", df_last["episode_key"].nunique())
print("Duplicated patients:", df_last["pubID"].duplicated().sum())
print("Duplicated episodes:", df_last["episode_key"].duplicated().sum())

assert df_last["pubID"].duplicated().sum() == 0
assert df_last["episode_key"].duplicated().sum() == 0

flow_summary = pd.DataFrame(flow_rows)
display(flow_summary)

In [ ]:
# ============================================================
# TEMPORAL COHORTS
# ============================================================

df_last["temporal_cohort"] = pd.Series(pd.NA, index=df_last.index, dtype="string")

df_last.loc[
    (df_last["endDate"] >= COHORT1_START) &
    (df_last["endDate"] <= COHORT1_END),
    "temporal_cohort"
] = "Cohort 1"

df_last.loc[
    (df_last["endDate"] >= COHORT2_START) &
    (df_last["endDate"] <= COHORT2_END),
    "temporal_cohort"
] = "Cohort 2"

df_last = df_last[df_last["temporal_cohort"].isin(["Cohort 1", "Cohort 2"])].copy()

df_last["temporal_cohort"] = pd.Categorical(
    df_last["temporal_cohort"],
    categories=["Cohort 1", "Cohort 2"],
    ordered=True
)

df_last["admission_year"] = df_last["endDate"].dt.year

print(df_last["temporal_cohort"].value_counts(dropna=False))

In [ ]:
# ============================================================
# ADMISSION ORIGIN LABELS
# ============================================================

origin_map = {
    1: "Home (acute) or via Emergency Department",
    2: "Nursing home (acute) or via Emergency Department",
    3: "Other lower-intensity care wards",
    4: "Other higher-intensity care wards",
    5: "Intensive care unit",
    6: "Home (elective)"
}

df_last["origin_label"] = (
    pd.to_numeric(df_last["origin"], errors="coerce")
    .map(origin_map)
)

In [ ]:
# ============================================================
# QC: TEMPORAL SPLIT
# ============================================================

cohort1_max = df_last.loc[
    df_last["temporal_cohort"] == "Cohort 1",
    "endDate"
].max()

cohort2_min = df_last.loc[
    df_last["temporal_cohort"] == "Cohort 2",
    "endDate"
].min()

print("Cohort 1 max endDate:", cohort1_max)
print("Cohort 2 min endDate:", cohort2_min)

assert cohort1_max < cohort2_min

In [ ]:
# ============================================================
# CANCER AND METASTATIC GROUPS
# ============================================================

df_last["solidTumor"] = pd.to_numeric(
    df_last["solidTumor"],
    errors="coerce"
)

solid_tumor_map = {
    0: "No cancer",
    2: "Localized cancer",
    6: "Metastatic cancer",
}

df_last["solidTumor_cat"] = (
    df_last["solidTumor"]
    .map(solid_tumor_map)
    .fillna("Unknown")
)

df_last["metastatic_status"] = pd.Series(pd.NA, index=df_last.index, dtype="Int64")

df_last.loc[
    df_last["solidTumor_cat"] == "Metastatic cancer",
    "metastatic_status"
] = 1

df_last.loc[
    df_last["solidTumor_cat"].isin(["No cancer", "Localized cancer"]),
    "metastatic_status"
] = 0

df_last["metastatic_group"] = pd.Series(pd.NA, index=df_last.index, dtype="string")

df_last.loc[
    df_last["metastatic_status"] == 1,
    "metastatic_group"
] = "Metastatic"

df_last.loc[
    df_last["metastatic_status"] == 0,
    "metastatic_group"
] = "Non-metastatic"

print("Cancer status:")
print(df_last["solidTumor_cat"].value_counts(dropna=False))

print("\nMetastatic group:")
print(df_last["metastatic_group"].value_counts(dropna=False))

In [ ]:
# ============================================================
# OUTCOME DEFINITIONS
# ============================================================

df_last["dischargedAlive"] = pd.to_numeric(
    df_last["dischargedAlive"],
    errors="coerce"
)

df_last["dischargeMod"] = pd.to_numeric(
    df_last["dischargeMod"],
    errors="coerce"
)

# In-hospital mortality
df_last["in_hospital_mortality"] = np.nan

df_last.loc[df_last["dischargedAlive"] == 0, "in_hospital_mortality"] = 1
df_last.loc[df_last["dischargedAlive"] == 1, "in_hospital_mortality"] = 0

# Discharge to palliative care
df_last["discharge_to_palliative_care"] = np.nan

df_last.loc[
    df_last["dischargeMod"].isin(PALLIATIVE_DISCHARGE_CODES),
    "discharge_to_palliative_care"
] = 1

df_last.loc[
    df_last["dischargeMod"].notna() &
    ~df_last["dischargeMod"].isin(PALLIATIVE_DISCHARGE_CODES),
    "discharge_to_palliative_care"
] = 0

# Composite outcome: in-hospital mortality OR discharge to palliative care
# Missing-aware definition:
# - In-hospital death is always composite = 1.
# - Alive + palliative discharge is composite = 1.
# - Alive + non-palliative discharge is composite = 0.
# - Alive + missing discharge mode remains missing.
df_last["composite_inhospital_or_palliative"] = np.nan

df_last.loc[
    df_last["in_hospital_mortality"] == 1,
    "composite_inhospital_or_palliative"
] = 1

df_last.loc[
    (df_last["in_hospital_mortality"] == 0) &
    (df_last["discharge_to_palliative_care"] == 1),
    "composite_inhospital_or_palliative"
] = 1

df_last.loc[
    (df_last["in_hospital_mortality"] == 0) &
    (df_last["discharge_to_palliative_care"] == 0),
    "composite_inhospital_or_palliative"
] = 0

df_last["death_days_after_discharge"] = (
    df_last["dateOfDeath"] - df_last["endDate"]
).dt.days

In [ ]:
# ============================================================
# OUTCOME CONSISTENCY CHECKS
# ============================================================

death_before_admission_mask = (
    df_last["dateOfDeath"].notna() &
    (df_last["dateOfDeath"] < df_last["startDate"])
)

death_before_discharge_alive_mask = (
    (df_last["dischargedAlive"] == 1) &
    df_last["dateOfDeath"].notna() &
    (df_last["dateOfDeath"] < df_last["endDate"])
)

inhosp_death_without_death_date_mask = (
    (df_last["dischargedAlive"] == 0) &
    df_last["dateOfDeath"].isna()
)

print("Death before admission:", death_before_admission_mask.sum())
print("Death before discharge among discharged alive:", death_before_discharge_alive_mask.sum())
print("In-hospital death without dateOfDeath:", inhosp_death_without_death_date_mask.sum())

display(
    df_last.loc[
        death_before_admission_mask |
        death_before_discharge_alive_mask |
        inhosp_death_without_death_date_mask,
        [
            "pubID",
            "episode_key",
            "startDate",
            "endDate",
            "dischargedAlive",
            "dateOfDeath",
            "death_days_after_discharge",
            "hospitalWard",
            "temporal_cohort",
        ]
    ].sort_values(["pubID", "episode_key"])
)

In [ ]:
# ============================================================
# HANDLE INCONSISTENT DEATH DATES
# ============================================================

inconsistent_death_mask = (
    df_last["dateOfDeath"].notna() &
    (
        (df_last["dateOfDeath"] < df_last["startDate"]) |
        (
            (df_last["dischargedAlive"] == 1) &
            (df_last["dateOfDeath"] < df_last["endDate"])
        )
    )
)

print("Inconsistent death dates set to NaT:", inconsistent_death_mask.sum())

df_last.loc[inconsistent_death_mask, "dateOfDeath"] = pd.NaT

df_last["death_days_after_discharge"] = (
    df_last["dateOfDeath"] - df_last["endDate"]
).dt.days

In [ ]:
# ============================================================
# OUTCOME DEFINITIONS: 90-DAY POST-DISCHARGE MORTALITY
# ============================================================

df_last["has_90d_followup"] = np.where(
    (df_last["dischargedAlive"] == 1) &
    (df_last["endDate"] + pd.Timedelta(days=90) <= FOLLOWUP_DATE),
    1,
    0
)

eligible_90d = (
    (df_last["dischargedAlive"] == 1) &
    (df_last["has_90d_followup"] == 1)
)

df_last["mortality_90d_post_discharge"] = np.nan

df_last.loc[eligible_90d, "mortality_90d_post_discharge"] = np.where(
    df_last.loc[eligible_90d, "death_days_after_discharge"].between(0, 90),
    1,
    0
)

print("Outcome summary:")
for col in [
    "in_hospital_mortality",
    "discharge_to_palliative_care",
    "composite_inhospital_or_palliative",
    "mortality_90d_post_discharge",
]:
    print(col, ":", n_pct(df_last[col]))

print("\nFollow-up availability:")
print(df_last["has_90d_followup"].value_counts(dropna=False))

In [ ]:
# ============================================================
# SURVIVAL-READY VARIABLES
# ============================================================

# Survival analysis is defined among patients discharged alive.
survival_eligible = (
    (df_last["dischargedAlive"] == 1) &
    (df_last["has_90d_followup"] == 1)
)

df_last["survival_time_days"] = np.nan
df_last["survival_time_90d"] = np.nan
df_last["survival_event_90d"] = np.nan

df_last.loc[survival_eligible, "survival_time_days"] = np.where(
    df_last.loc[survival_eligible, "dateOfDeath"].notna(),
    (
        df_last.loc[survival_eligible, "dateOfDeath"] -
        df_last.loc[survival_eligible, "endDate"]
    ).dt.days,
    (
        FOLLOWUP_DATE -
        df_last.loc[survival_eligible, "endDate"]
    ).dt.days
)

df_last.loc[survival_eligible, "survival_time_90d"] = (
    df_last.loc[survival_eligible, "survival_time_days"]
    .clip(lower=0, upper=90)
)

df_last.loc[survival_eligible, "survival_event_90d"] = np.where(
    df_last.loc[survival_eligible, "death_days_after_discharge"].between(0, 90),
    1,
    0
)

print("Survival eligible:", survival_eligible.sum())
print("90-day survival events:", int(df_last["survival_event_90d"].sum(skipna=True)))

In [ ]:
# ============================================================
# SELECT FINAL COHORT VARIABLES
# ============================================================

baseline_model_variables = [
    # demographics
    "age",
    "sex",
    "bmi",

    # ward / admission
    "hospitalWard",
    "origin",
    "daysInER",
    "daysInER_capped",
    "LOS_days",
    "admLast12m",
    "admLast6m",

    # functional / severity
    "admECOG",
    "admBarthelScore",
    "admBradenScore",
    "adm_mews",
    "adm_gcsScore",

    # comorbidity
    "cirsTotalScore",
    "cirs_CI",
    "cirs_SI",
    "ccsi",

    # oncology
    "solidTumor_cat",
    "metastatic_status",
    "metastatic_group",
]

identifier_cols = [
    "CloudPatientId",
    "episode_key",
    "pubID",
    "episodeID",
    "hospitalWard",
    "startDate",
    "endDate",
    "LOS_days",
    "daysInER",
    "daysInER_capped",
    "ER_last_day",
    "ward_startDate",
    "dateOfDeath",
    "temporal_cohort",
    "admission_year",
]

outcome_cols = [
    "in_hospital_mortality",
    "discharge_to_palliative_care",
    "composite_inhospital_or_palliative",
    "mortality_90d_post_discharge",
    "has_90d_followup",
    "death_days_after_discharge",
    "survival_time_days",
    "survival_time_90d",
    "survival_event_90d",
]

extra_clinical_cols = [
    "dischargedAlive",
    "dischargeMod",
    "solidTumor",
    "solidTumor_cat",
    "metastatic_status",
    "metastatic_group",
]

final_cols = (
    identifier_cols
    + outcome_cols
    + baseline_model_variables
    + extra_clinical_cols
)

final_cols = list(dict.fromkeys([c for c in final_cols if c in df_last.columns]))

df_final_cohort = df_last[final_cols].copy()
df_final_cohort = remove_duplicate_columns(df_final_cohort)

print("Duplicated columns:", df_final_cohort.columns[df_final_cohort.columns.duplicated()].tolist())

print("02_final_cohort:", df_final_cohort.shape)
print("\nCIRS columns:")
print([c for c in df_final_cohort.columns if "cirs" in c.lower()])
print("Patients:", df_final_cohort["pubID"].nunique())
print("Episodes:", df_final_cohort["episode_key"].nunique())

display(df_final_cohort.head())

In [ ]:
# ============================================================
# BASELINE VARIABLE MISSINGNESS
# ============================================================

missing_summary = (
    df_final_cohort
    .isna()
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
)

missing_summary.columns = ["variable", "missing_pct"]

missing_summary = missing_summary.sort_values(
    "missing_pct",
    ascending=False
)

display(missing_summary)

In [ ]:
# ============================================================
# QC: SHARED MISSINGNESS PATTERN
# ============================================================

core_missing_vars = [
    "age",
    "sex",
    "ccsi",
    "solidTumor",
]

missing_pattern = (
    df_final_cohort[core_missing_vars]
    .isna()
    .astype(int)
)

missing_pattern["n_missing"] = missing_pattern.sum(axis=1)

print(
    missing_pattern["n_missing"]
    .value_counts()
    .sort_index()
)

display(
    df_final_cohort.loc[
        missing_pattern["n_missing"] >= 3,
        [
            "pubID",
            "episode_key",
            "hospitalWard",
            "temporal_cohort",
            "age",
            "sex",
            "ccsi",
            "solidTumor",
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# BASELINE TABLE SPECIFICATIONS
# ============================================================

table1_specs = [
    # section, variable, label, type
    ("Demographics", "age", "Age (years)", "continuous"),
    ("Demographics", "bmi", "BMI", "continuous"),
    ("Demographics", "sex", "Female sex, n (%)", "binary_female"),

    ("Comorbidity burden", "cirs_CI", "CIRS Comorbidity Index", "continuous"),
    ("Comorbidity burden", "cirs_SI", "CIRS Severity Index", "continuous"),
    ("Comorbidity burden", "ccsi", "Charlson Comorbidity Index", "continuous"),

    ("Functional status", "admBarthelScore", "Barthel Index at admission", "continuous"),
    ("Functional status", "admBradenScore", "Braden score at admission", "continuous"),
    ("Functional status", "admECOG", "ECOG performance status at admission", "continuous"),

    ("Admission severity", "adm_mews", "MEWS", "continuous"),
    ("Admission severity", "adm_gcsScore", "GCS score at admission", "continuous"),

    ("Healthcare utilization", "admLast6m", "Admissions in previous 6 months", "continuous"),
    ("Healthcare utilization", "admLast12m", "Admissions in previous 12 months", "continuous"),

    ("Hospitalization", "daysInER", "Days in emergency department", "continuous"),
    ("Hospitalization", "LOS_days", "Length of stay (days)", "continuous"),

    ("Hospital ward", "hospitalWard", "Hospital ward", "categorical"),
    ("Admission origin", "origin_label", "Admission origin", "categorical"),

    ("Oncological status", "solidTumor_cat", "Solid tumor status", "categorical"),
    ("Oncological status", "metastatic_group", "Metastatic status", "categorical"),

    ("Outcome", "in_hospital_mortality", "In-hospital mortality, n (%)", "binary_1"),
    ("Outcome", "composite_inhospital_or_palliative", "Composite outcome, n (%)", "binary_1"),
    ("Outcome", "mortality_90d_post_discharge", "90-day post-discharge mortality, n (%)", "binary_1"),
]

table1_specs = [
    spec for spec in table1_specs
    if spec[1] in df_final_cohort.columns
]

In [ ]:
# ============================================================
# PUBLICATION-STYLE BASELINE TABLE FUNCTION
# ============================================================

def summarize_variable(df, var, var_type):
    if var_type == "continuous":
        return median_iqr(df[var])

    if var_type == "binary_1":
        return n_pct(df[var])

    if var_type == "binary_female":

        s = (
            df[var]
            .astype("string")
            .str.strip()
            .str.lower()
            .dropna()
        )

        if len(s) == 0:
            return "NA"

        female_mask = s.isin(["f", "female"])

        n = int(female_mask.sum())
        total = len(s)

        return f"{n} ({100*n/total:.1f}%)"

    if var_type == "categorical":
        return ""

    return "NA"


def make_publication_table(df, group_col=None):
    rows = []

    if group_col is not None:
        df = df[df[group_col].notna()].copy()
        groups = list(df[group_col].dropna().unique())
    else:
        groups = []

    for section, var, label, var_type in table1_specs:
        
        # Do not compare the grouping variable with itself
        if group_col is not None and var == group_col:
            continue

        if var_type != "categorical":
            row = {
                "Section": section,
                "Variable": label,
                "Overall": summarize_variable(df, var, var_type),
            }

            if group_col is not None:
                for g in groups:
                    row[str(g)] = summarize_variable(
                        df.loc[df[group_col] == g],
                        var,
                        var_type
                    )

                if var_type == "continuous":
                    p = compare_continuous(df, var, group_col)
                    smd = smd_continuous(df, var, group_col)
                else:
                    p = compare_categorical(df, var, group_col)
                    smd = smd_binary(df, var, group_col, positive_value=1)

                row["p-value"] = format_p(p)
                row["SMD"] = "" if pd.isna(smd) else f"{smd:.3f}"

            rows.append(row)

        else:
            categories = list(df[var].dropna().unique())

            p = compare_categorical(df, var, group_col) if group_col is not None else np.nan

            for i, cat in enumerate(categories):
                row = {
                    "Section": section if i == 0 else "",
                    "Variable": str(cat),
                    "Overall": categorical_n_pct(df[var], cat),
                }

                if group_col is not None:
                    for g in groups:
                        row[str(g)] = categorical_n_pct(
                            df.loc[df[group_col] == g, var],
                            cat
                        )

                    row["p-value"] = format_p(p) if i == 0 else ""
                    row["SMD"] = ""

                rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# BASELINE TABLES
# ============================================================

baseline_overall = make_publication_table(
    df_final_cohort,
    group_col=None
)

baseline_temporal = make_publication_table(
    df_final_cohort,
    group_col="temporal_cohort"
)

baseline_metastatic = make_publication_table(
    df_final_cohort[df_final_cohort["metastatic_group"].notna()].copy(),
    group_col="metastatic_group"
)

baseline_ward = make_publication_table(
    df_final_cohort,
    group_col="hospitalWard"
)

display(baseline_overall.head(20))
display(baseline_temporal.head(20))
display(baseline_metastatic.head(20))
display(baseline_ward.head(20))

In [ ]:
# ============================================================
# OUTCOMES SUMMARY
# ============================================================

primary_outcomes = [
    "composite_inhospital_or_palliative",
    "mortality_90d_post_discharge",
]

all_outcomes = [
    "in_hospital_mortality",
    "discharge_to_palliative_care",
    "composite_inhospital_or_palliative",
    "mortality_90d_post_discharge",
]

summary_rows = []

for outcome in all_outcomes:
    row = {
        "outcome": outcome,
        "overall": n_pct(df_final_cohort[outcome]),
        "Cohort 1": n_pct(df_final_cohort.loc[df_final_cohort["temporal_cohort"] == "Cohort 1", outcome]),
        "Cohort 2": n_pct(df_final_cohort.loc[df_final_cohort["temporal_cohort"] == "Cohort 2", outcome]),
        "Non-metastatic": n_pct(df_final_cohort.loc[df_final_cohort["metastatic_group"] == "Non-metastatic", outcome]),
        "Metastatic": n_pct(df_final_cohort.loc[df_final_cohort["metastatic_group"] == "Metastatic", outcome]),
        "Ward A": n_pct(df_final_cohort.loc[df_final_cohort["hospitalWard"] == "A", outcome]),
        "Ward B": n_pct(df_final_cohort.loc[df_final_cohort["hospitalWard"] == "B", outcome]),
    }
    summary_rows.append(row)

outcomes_summary = pd.DataFrame(summary_rows)

display(outcomes_summary)

In [ ]:
# ============================================================
# FINAL COHORT SUMMARY
# ============================================================

print("=" * 80)
print("02 COHORT DEFINITION SUMMARY")
print("=" * 80)

print("\nFinal cohort")
print("Rows:", df_final_cohort.shape[0])
print("Patients:", df_final_cohort["pubID"].nunique())
print("Episodes:", df_final_cohort["episode_key"].nunique())

print("\nTemporal cohorts:")
print(df_final_cohort["temporal_cohort"].value_counts(dropna=False))

print("\nHospital wards:")
print(df_final_cohort["hospitalWard"].value_counts(dropna=False))

print("\nCancer status:")
print(df_final_cohort["solidTumor_cat"].value_counts(dropna=False))

print("\nMetastatic group:")
print(df_final_cohort["metastatic_group"].value_counts(dropna=False))

print("\nFollow-up availability:")
print(df_final_cohort["has_90d_followup"].value_counts(dropna=False))

print("\nPrimary outcomes:")
for outcome in primary_outcomes:
    print(outcome, ":", n_pct(df_final_cohort[outcome]))

print("\nSurvival variables:")
print("Eligible for 90-day survival:", df_final_cohort["survival_time_90d"].notna().sum())
print("90-day survival events:", int(df_final_cohort["survival_event_90d"].sum(skipna=True)))

assert df_final_cohort["pubID"].duplicated().sum() == 0
assert df_final_cohort["episode_key"].duplicated().sum() == 0

In [ ]:
# ============================================================
# QC: DISCHARGED ALIVE COMPLETENESS
# ============================================================

print("dischargedAlive value counts:")
print(df_final_cohort["dischargedAlive"].value_counts(dropna=False))

print("\nMissing dischargedAlive:")
print(df_final_cohort["dischargedAlive"].isna().sum())

assert df_final_cohort["dischargedAlive"].isna().sum() == 0, \
    "dischargedAlive has missing values; in-hospital mortality definition needs review."

In [ ]:
# ============================================================
# SAVE OUTPUTS
# ============================================================

df_final_cohort.to_parquet(
    OUTPUT_02 / "02_final_cohort.parquet",
    index=False
)

flow_summary.to_excel(
    OUTPUT_02 / "02_flow_summary.xlsx",
    index=False
)

missing_summary.to_excel(
    OUTPUT_02 / "02_missingness_summary.xlsx",
    index=False
)

with pd.ExcelWriter(OUTPUT_02 / "02_baseline_table.xlsx") as writer:
    baseline_overall.to_excel(writer, sheet_name="overall", index=False)
    baseline_temporal.to_excel(writer, sheet_name="temporal_cohorts", index=False)
    baseline_metastatic.to_excel(writer, sheet_name="metastatic_status", index=False)
    baseline_ward.to_excel(writer, sheet_name="ward_sensitivity", index=False)

outcomes_summary.to_excel(
    OUTPUT_02 / "02_outcomes_summary.xlsx",
    index=False
)

print("Saved:")
print(OUTPUT_02 / "02_final_cohort.parquet")
print(OUTPUT_02 / "02_flow_summary.xlsx")
print(OUTPUT_02 / "02_missingness_summary.xlsx")
print(OUTPUT_02 / "02_baseline_table.xlsx")
print(OUTPUT_02 / "02_outcomes_summary.xlsx")